# Legal Contract Analyzer — Colab API Runner

This notebook runs the FastAPI backend in Colab and lets you call it via curl-like requests.

Notes:
- Llama 3.1 is gated; accept the license on Hugging Face and set `LEGAL_ANALYZER_HF_TOKEN`.
- Colab is GPU-capable and much faster than local CPU for Llama 3.1.


In [ ]:
# Clone the repo
import os
import pathlib
import subprocess

REPO_URL = 'https://github.com/YOUR_ORG/YOUR_REPO.git'
REPO_DIR = 'legal-contract-analyzer'

if not pathlib.Path(REPO_DIR).exists():
    if 'YOUR_REPO' in REPO_URL:
        raise ValueError('Set REPO_URL to your repo first.')
    subprocess.run(['git', 'clone', REPO_URL], check=True)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())


In [ ]:
# Install dependencies
!pip -q install -r requirements.txt


In [ ]:
# Configure notebook-like pipeline
import os

os.environ['LEGAL_ANALYZER_NOTEBOOK_MODE'] = 'true'
os.environ['LEGAL_ANALYZER_DEFAULT_TOP_K'] = '5'
os.environ['LEGAL_ANALYZER_LLM_MODEL'] = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
os.environ['LEGAL_ANALYZER_HF_TOKEN'] = 'YOUR_HF_TOKEN'  # required for Llama 3.1


In [ ]:
# Start API in the background
!uvicorn app.main:app --host 0.0.0.0 --port 8000 --log-level info &


In [ ]:
# Upload a PDF
from google.colab import files
uploaded = files.upload()
pdf_path = next(iter(uploaded))
print('Uploaded:', pdf_path)


In [ ]:
# Upload to API and capture document_id
import json
import subprocess

resp = subprocess.check_output([
    'curl', '-sS', '-X', 'POST', 'http://127.0.0.1:8000/upload',
    '-F', f'file=@{pdf_path}',
]).decode()
print(resp)
doc_id = json.loads(resp)['document_id']
print('document_id:', doc_id)


In [ ]:
# Ask a question
import json
import subprocess

payload = {
    'document_id': doc_id,
    'query': 'Who are the contract-holders?',
    'top_k': 6,
}
resp = subprocess.check_output([
    'curl', '-sS', '-X', 'POST', 'http://127.0.0.1:8000/analyze',
    '-H', 'Content-Type: application/json',
    '-d', json.dumps(payload),
]).decode()
print(resp)


In [ ]:
# Summary (optional)
import subprocess
resp = subprocess.check_output([
    'curl', '-sS', f'http://127.0.0.1:8000/summary/{doc_id}',
]).decode()
print(resp)
